# MediaPipe Landmark Extractor
## ADSP 32023 - Advanced Computer Vision Using Deep Learning
## Spring 2026

In [ ]:
# ============================================================
# 0. Install MediaPipe-only environment
# ============================================================
!pip uninstall -y tensorflow tf-keras tensorflow-text mediapipe protobuf -q
!pip install -q mediapipe==0.10.14 protobuf==4.25.3 kagglehub opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 24.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydf-tf 2.20.0 requires tensorflow==2.20.0, which is not installed.
tensorflow-hub 0.16.1 requires tf-keras>=2.14.1, which is not installed.
google-cloud-bigquery-storage 2.37.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
google-cloud-trace 1.19.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
grain 0.2.16 requires protobuf>=5.28.3, but you have protobuf 4.25.3 which is incompatible.
google-cloud-resource-manager 1.17.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 4.25.3 which is incompatible.
google-cloud-dataproc 5.27.0 requires protobuf<8.0.0,>=4.25.8, but you have

In [ ]:
# ============================================================
# 1. Imports
# ============================================================
import os
import cv2
import json
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from pathlib import Path
from google.colab import drive

import mediapipe as mp

print("MediaPipe version:", mp.__version__)
print("Has solutions:", hasattr(mp, "solutions"))

MediaPipe version: 0.10.14
Has solutions: True


In [ ]:
# ============================================================
# 2. Config
# ============================================================
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/UChicago/Masters/Spring/ADSP 32023/Computer Vision Final Project/data/preprocessed/"

os.makedirs(SAVE_DIR, exist_ok=True)

NUM_FRAMES = 48
SEQUENCE_LENGTH = NUM_FRAMES
STEP_SIZE = NUM_FRAMES // 2

REMOVE_PLANK = True

print("SAVE_DIR:", SAVE_DIR)

Mounted at /content/drive
SAVE_DIR: /content/drive/MyDrive/UChicago/Masters/Spring/Computer Vision/Computer Vision Final Project/mediapipe_pose_exercise_new_dataset/


In [ ]:
# ============================================================
# 3. Download dataset
# ============================================================
path = kagglehub.dataset_download("philosopher0808/gym-workoutexercises-video")
DATASET_PATH = Path(path)

print("Dataset path:", DATASET_PATH)

print("\nTop-level folders:")
for item in sorted(DATASET_PATH.iterdir()):
    print("-", item.name)

100%|██████████| 9.63G/9.63G [04:28<00:00, 38.6MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/philosopher0808/gym-workoutexercises-video/versions/1

Top-level folders:
- raw_data
- test
- verified_data


In [ ]:
# ============================================================
# 4. Collect only verified_data and test videos
# ============================================================
VIDEO_EXTS = (".mp4", ".mov", ".avi", ".mkv", ".webm")

rows = []

for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.lower().endswith(VIDEO_EXTS):
            file_path = Path(root) / file
            rel_parts = file_path.relative_to(DATASET_PATH).parts

            top_level = rel_parts[0] if len(rel_parts) > 0 else "unknown"

            if top_level not in ["verified_data", "test"]:
                continue

            exercise = rel_parts[-2].strip()

            rows.append({
                "filepath": str(file_path),
                "filename": file,
                "top_level": top_level,
                "exercise": exercise
            })

df = pd.DataFrame(rows)

if REMOVE_PLANK:
    df = df[df["exercise"] != "plank"].reset_index(drop=True)

print("Total verified/test videos:", len(df))
print("Number of exercises:", df["exercise"].nunique())

print("\nVideos by split:")
display(df["top_level"].value_counts().to_frame("num_videos"))

print("\nVideos by exercise and split:")
exercise_split_counts = (
    df.groupby(["exercise", "top_level"])
      .size()
      .unstack(fill_value=0)
      .rename(columns={
          "verified_data": "verified_videos",
          "test": "test_videos"
      })
)

exercise_split_counts["total_videos"] = exercise_split_counts.sum(axis=1)
exercise_split_counts = exercise_split_counts.sort_values("total_videos", ascending=False)

display(exercise_split_counts)
display(df.head())

Total verified/test videos: 1479
Number of exercises: 21

Videos by split:


,num_videos
top_level,
verified_data,1420
test,59



Videos by exercise and split:


top_level,test_videos,verified_videos,total_videos
exercise,,,
barbell biceps curl,2,111,113
bench press,3,94,97
squat,3,87,90
push-up,2,83,85
decline bench press,3,81,84
chest fly machine,4,76,80
hammer curl,3,73,76
incline bench press,3,72,75
deadlift,2,69,71


,filepath,filename,top_level,exercise
0,/root/.cache/kagglehub/datasets/philosopher080...,cee97ae4-697f-4eba-9e48-e668f0568e3a.mp4,verified_data,shoulder press
1,/root/.cache/kagglehub/datasets/philosopher080...,6397dc79-5ba3-4244-a426-c24d6e7cfeb0.mp4,verified_data,shoulder press
2,/root/.cache/kagglehub/datasets/philosopher080...,30fe126e-0985-48af-813f-65c30d49913d.mp4,verified_data,shoulder press
3,/root/.cache/kagglehub/datasets/philosopher080...,b066cbc1-931a-4af5-b668-36e98e7091b6.mp4,verified_data,shoulder press
4,/root/.cache/kagglehub/datasets/philosopher080...,17496a37-c1f7-4105-9c07-e7cfbca06106.mp4,verified_data,shoulder press


In [ ]:
# ============================================================
# 5. Split verified and test metadata
# ============================================================
verified_df = df[df["top_level"] == "verified_data"].reset_index(drop=True)
test_df = df[df["top_level"] == "test"].reset_index(drop=True)

print("Verified videos:", len(verified_df))
print("Test videos:", len(test_df))
print("Verified classes:", verified_df["exercise"].nunique())
print("Test classes:", test_df["exercise"].nunique())

Verified videos: 1420
Test videos: 59
Verified classes: 21
Test classes: 21


In [ ]:
# ============================================================
# 6. Encode labels based on verified_data only
# ============================================================
label_names = sorted(verified_df["exercise"].unique())
label_to_idx = {label: i for i, label in enumerate(label_names)}
idx_to_label = {i: label for label, i in label_to_idx.items()}

verified_df["label"] = verified_df["exercise"].map(label_to_idx)

# Keep only test classes that also exist in verified data
test_df = test_df[test_df["exercise"].isin(label_to_idx.keys())].reset_index(drop=True)
test_df["label"] = test_df["exercise"].map(label_to_idx)

LABEL_PATH = os.path.join(SAVE_DIR, "label_names.json")
LABEL_MAP_PATH = os.path.join(SAVE_DIR, "label_to_idx.json")

with open(LABEL_PATH, "w") as f:
    json.dump(label_names, f)

with open(LABEL_MAP_PATH, "w") as f:
    json.dump(label_to_idx, f, indent=2)

print("Saved labels to:", LABEL_PATH)
print("Number of classes:", len(label_names))
print(label_to_idx)

print("\nFinal verified videos:", len(verified_df))
print("Final test videos:", len(test_df))

Saved labels to: /content/drive/MyDrive/UChicago/Masters/Spring/Computer Vision/Computer Vision Final Project/mediapipe_pose_exercise_new_dataset/label_names.json
Number of classes: 21
{'barbell biceps curl': 0, 'bench press': 1, 'chest fly machine': 2, 'deadlift': 3, 'decline bench press': 4, 'hammer curl': 5, 'hip thrust': 6, 'incline bench press': 7, 'lat pulldown': 8, 'lateral raise': 9, 'leg extension': 10, 'leg raises': 11, 'pull Up': 12, 'push-up': 13, 'romanian deadlift': 14, 'russian twist': 15, 'shoulder press': 16, 'squat': 17, 't bar row': 18, 'tricep Pushdown': 19, 'tricep dips': 20}

Final verified videos: 1420
Final test videos: 59


In [ ]:
# ============================================================
# 7. MediaPipe setup
# ============================================================
mp_pose = mp.solutions.pose

pose_detector = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    enable_segmentation=False,
    min_detection_confidence=0.4,
    min_tracking_confidence=0.4
)

In [ ]:
# ============================================================
# 8. MediaPipe landmark indices
# ============================================================
NOSE = 0
LEFT_SHOULDER = 11
RIGHT_SHOULDER = 12
LEFT_ELBOW = 13
RIGHT_ELBOW = 14
LEFT_WRIST = 15
RIGHT_WRIST = 16
LEFT_HIP = 23
RIGHT_HIP = 24
LEFT_KNEE = 25
RIGHT_KNEE = 26
LEFT_ANKLE = 27
RIGHT_ANKLE = 28
LEFT_HEEL = 29
RIGHT_HEEL = 30
LEFT_FOOT_INDEX = 31
RIGHT_FOOT_INDEX = 32

In [ ]:
# ============================================================
# 9. Feature helpers
# Base frame feature size:
#   33 landmarks * 4 = 132
#   11 angle features = 11
#   7 geometry features = 7
#   total base features = 150
#
# Final feature size:
#   150 base + 150 velocity = 300
# ============================================================

def calculate_angle(a, b, c):
    a = np.array(a, dtype=np.float32)
    b = np.array(b, dtype=np.float32)
    c = np.array(c, dtype=np.float32)

    ba = a - b
    bc = c - b

    denom = np.linalg.norm(ba) * np.linalg.norm(bc)

    if denom == 0:
        return 0.0

    cosine = np.dot(ba, bc) / denom
    cosine = np.clip(cosine, -1.0, 1.0)

    return np.degrees(np.arccos(cosine))


def landmarks_to_np(results):
    if results.pose_landmarks is None:
        return None

    landmarks = []

    for lm in results.pose_landmarks.landmark:
        landmarks.append([lm.x, lm.y, lm.z, lm.visibility])

    return np.array(landmarks, dtype=np.float32)


def normalize_landmarks(lms):
    lms = np.array(lms, dtype=np.float32)

    left_hip = lms[LEFT_HIP, :3]
    right_hip = lms[RIGHT_HIP, :3]
    left_shoulder = lms[LEFT_SHOULDER, :3]
    right_shoulder = lms[RIGHT_SHOULDER, :3]

    hip_center = (left_hip + right_hip) / 2.0
    shoulder_center = (left_shoulder + right_shoulder) / 2.0

    torso_size = np.linalg.norm(shoulder_center - hip_center)

    if torso_size < 1e-6:
        torso_size = 1.0

    normalized_xyz = (lms[:, :3] - hip_center) / torso_size
    visibility = lms[:, 3:4]

    return np.concatenate([normalized_xyz, visibility], axis=1)


def mediapipe_landmarks_to_base_features(lms):
    normalized = normalize_landmarks(lms)
    flat_landmarks = normalized.flatten()

    xyz = lms[:, :3]

    angle_features = []

    # Elbows
    angle_features.append(calculate_angle(xyz[LEFT_SHOULDER], xyz[LEFT_ELBOW], xyz[LEFT_WRIST]))
    angle_features.append(calculate_angle(xyz[RIGHT_SHOULDER], xyz[RIGHT_ELBOW], xyz[RIGHT_WRIST]))

    # Shoulders
    angle_features.append(calculate_angle(xyz[LEFT_ELBOW], xyz[LEFT_SHOULDER], xyz[LEFT_HIP]))
    angle_features.append(calculate_angle(xyz[RIGHT_ELBOW], xyz[RIGHT_SHOULDER], xyz[RIGHT_HIP]))

    # Knees
    angle_features.append(calculate_angle(xyz[LEFT_HIP], xyz[LEFT_KNEE], xyz[LEFT_ANKLE]))
    angle_features.append(calculate_angle(xyz[RIGHT_HIP], xyz[RIGHT_KNEE], xyz[RIGHT_ANKLE]))

    # Hips
    angle_features.append(calculate_angle(xyz[LEFT_SHOULDER], xyz[LEFT_HIP], xyz[LEFT_KNEE]))
    angle_features.append(calculate_angle(xyz[RIGHT_SHOULDER], xyz[RIGHT_HIP], xyz[RIGHT_KNEE]))

    # Ankles
    angle_features.append(calculate_angle(xyz[LEFT_KNEE], xyz[LEFT_ANKLE], xyz[LEFT_FOOT_INDEX]))
    angle_features.append(calculate_angle(xyz[RIGHT_KNEE], xyz[RIGHT_ANKLE], xyz[RIGHT_FOOT_INDEX]))

    # Torso angle relative to vertical
    hip_center = (xyz[LEFT_HIP] + xyz[RIGHT_HIP]) / 2.0
    shoulder_center = (xyz[LEFT_SHOULDER] + xyz[RIGHT_SHOULDER]) / 2.0

    torso_vec = shoulder_center[:2] - hip_center[:2]
    vertical_vec = np.array([0, -1], dtype=np.float32)

    denom = np.linalg.norm(torso_vec) * np.linalg.norm(vertical_vec)

    if denom == 0:
        torso_angle = 0.0
    else:
        torso_angle = np.degrees(
            np.arccos(
                np.clip(np.dot(torso_vec, vertical_vec) / denom, -1.0, 1.0)
            )
        )

    angle_features.append(torso_angle)

    angle_features = np.array(angle_features, dtype=np.float32) / 180.0

    # Geometry features
    shoulder_width = np.linalg.norm(xyz[LEFT_SHOULDER] - xyz[RIGHT_SHOULDER])
    hip_width = np.linalg.norm(xyz[LEFT_HIP] - xyz[RIGHT_HIP])
    body_height = np.linalg.norm(shoulder_center - hip_center)

    wrist_y_mean = (xyz[LEFT_WRIST, 1] + xyz[RIGHT_WRIST, 1]) / 2.0
    shoulder_y_mean = (xyz[LEFT_SHOULDER, 1] + xyz[RIGHT_SHOULDER, 1]) / 2.0
    hip_y_mean = (xyz[LEFT_HIP, 1] + xyz[RIGHT_HIP, 1]) / 2.0
    knee_y_mean = (xyz[LEFT_KNEE, 1] + xyz[RIGHT_KNEE, 1]) / 2.0

    wrist_vs_shoulder = wrist_y_mean - shoulder_y_mean
    wrist_vs_hip = wrist_y_mean - hip_y_mean
    knee_vs_hip = knee_y_mean - hip_y_mean

    wrist_distance = abs(xyz[LEFT_WRIST, 0] - xyz[RIGHT_WRIST, 0])

    geometry_features = np.array([
        shoulder_width,
        hip_width,
        body_height,
        wrist_vs_shoulder,
        wrist_vs_hip,
        knee_vs_hip,
        wrist_distance
    ], dtype=np.float32)

    features = np.concatenate(
        [flat_landmarks, angle_features, geometry_features],
        axis=0
    ).astype(np.float32)

    if features.shape[0] < 150:
        features = np.pad(features, (0, 150 - features.shape[0]))
    elif features.shape[0] > 150:
        features = features[:150]

    return features.astype(np.float32)


def add_velocity_features(seq):
    velocity = np.diff(seq, axis=0, prepend=seq[:1])
    return np.concatenate([seq, velocity], axis=1).astype(np.float32)

In [ ]:
# ============================================================
# 10. Extract frame-level features from one video
# ============================================================

def extract_all_frame_features(video_path):
    cap = cv2.VideoCapture(video_path)

    frame_features = []
    last_features = None

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose_detector.process(frame_rgb)

        lms = landmarks_to_np(results)

        if lms is None:
            if last_features is not None:
                frame_features.append(last_features)
            else:
                frame_features.append(None)
            continue

        features = mediapipe_landmarks_to_base_features(lms)
        frame_features.append(features)
        last_features = features

    cap.release()

    feature_dim = 150
    cleaned = []

    for f in frame_features:
        if f is None:
            cleaned.append(np.zeros(feature_dim, dtype=np.float32))
        else:
            cleaned.append(f.astype(np.float32))

    return cleaned

In [ ]:
# ============================================================
# 11. Make sliding-window sequences
# ============================================================

def make_sliding_window_sequences(
    frame_features,
    sequence_length=SEQUENCE_LENGTH,
    step_size=STEP_SIZE
):
    sequences = []

    if len(frame_features) == 0:
        return sequences

    if len(frame_features) < sequence_length:
        padded = list(frame_features)

        while len(padded) < sequence_length:
            padded.append(padded[-1])

        seq = np.stack(padded[:sequence_length]).astype(np.float32)
        seq = add_velocity_features(seq)
        sequences.append(seq)

        return sequences

    for start in range(0, len(frame_features) - sequence_length + 1, step_size):
        window = frame_features[start:start + sequence_length]

        seq = np.stack(window).astype(np.float32)
        seq = add_velocity_features(seq)

        sequences.append(seq)

    return sequences


def extract_mediapipe_sequences_from_video(video_path):
    frame_features = extract_all_frame_features(video_path)

    sequences = make_sliding_window_sequences(
        frame_features,
        sequence_length=SEQUENCE_LENGTH,
        step_size=STEP_SIZE
    )

    return sequences

In [ ]:
# ============================================================
# 12. Build MediaPipe sequence dataset
# ============================================================

def build_mediapipe_dataset(df_input, split_name):
    X, y, kept_paths, kept_labels, sequence_ids, filenames = [], [], [], [], [], []

    for _, row in tqdm(
        df_input.iterrows(),
        total=len(df_input),
        desc=f"Extracting MediaPipe pose: {split_name}"
    ):
        sequences = extract_mediapipe_sequences_from_video(row["filepath"])

        if len(sequences) == 0:
            continue

        for seq_id, seq in enumerate(sequences):
            X.append(seq)
            y.append(row["label"])
            kept_paths.append(row["filepath"])
            kept_labels.append(row["exercise"])
            sequence_ids.append(seq_id)
            filenames.append(row["filename"])

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int32)

    kept_df = pd.DataFrame({
        "filepath": kept_paths,
        "filename": filenames,
        "exercise": kept_labels,
        "label": y,
        "sequence_id": sequence_ids,
        "split": split_name
    })

    return X, y, kept_df

In [ ]:
# ============================================================
# 13. Extract verified and test features
# ============================================================

X_verified, y_verified, kept_verified_df = build_mediapipe_dataset(
    verified_df,
    split_name="verified"
)

X_test, y_test, kept_test_df = build_mediapipe_dataset(
    test_df,
    split_name="test"
)

print("X_verified shape:", X_verified.shape)
print("y_verified shape:", y_verified.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

print("\nVerified sequences per exercise:")
display(kept_verified_df["exercise"].value_counts().to_frame("num_sequences"))

print("\nTest sequences per exercise:")
display(kept_test_df["exercise"].value_counts().to_frame("num_sequences"))

Extracting MediaPipe pose: verified:   0%|          | 0/1420 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
Extracting MediaPipe pose: verified:  29%|██▉       | 412/1420 [2:06:54<5:10:30, 18.48s/it]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# 14. Save extracted features
# ============================================================

# Main verified/train files
np.save(os.path.join(SAVE_DIR, "X_mediapipe.npy"), X_verified)
np.save(os.path.join(SAVE_DIR, "y_all.npy"), y_verified)
kept_verified_df.to_csv(
    os.path.join(SAVE_DIR, "kept_mediapipe_dataset.csv"),
    index=False
)

# Explicit verified files
np.save(os.path.join(SAVE_DIR, "X_verified_mediapipe.npy"), X_verified)
np.save(os.path.join(SAVE_DIR, "y_verified.npy"), y_verified)
kept_verified_df.to_csv(
    os.path.join(SAVE_DIR, "kept_verified_mediapipe_dataset.csv"),
    index=False
)

# Test files
np.save(os.path.join(SAVE_DIR, "X_test_mediapipe.npy"), X_test)
np.save(os.path.join(SAVE_DIR, "y_test.npy"), y_test)
kept_test_df.to_csv(
    os.path.join(SAVE_DIR, "kept_test_mediapipe_dataset.csv"),
    index=False
)

metadata = {
    "dataset": "philosopher0808/gym-workoutexercises-video",
    "training_split": "verified_data",
    "test_split": "test",
    "removed_plank": REMOVE_PLANK,
    "num_frames": NUM_FRAMES,
    "sequence_length": SEQUENCE_LENGTH,
    "step_size": STEP_SIZE,
    "base_feature_dim": 150,
    "feature_dim": int(X_verified.shape[-1]),
    "num_classes": len(label_names),
    "label_names": label_names,
    "uses_sliding_windows": True,
    "uses_velocity_features": True,
    "verified_sequences": int(len(X_verified)),
    "test_sequences": int(len(X_test))
}

with open(os.path.join(SAVE_DIR, "feature_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved all MediaPipe features to:", SAVE_DIR)
print(metadata)

NameError: name 'np' is not defined